# 02 — Auditoría de calidad de datos

## StreamView Analytics · EP1

**Pregunta guía:** ¿qué problemas reales de calidad existen en Movies y TV Shows y qué decisión de tratamiento corresponde a cada uno?

Este notebook sigue detectar → medir → investigar → documentar → proponer decisión → justificar. Los CSV RAW se leen solo en memoria: no se modifican, no se generan datos procesados y no se aplican transformaciones finales.

## 1. Relación con la exploración previa

La exploración inicial ya identificó señales en rating, duration, show_id, créditos, campos multivalor, fechas y finanzas. Esta auditoría mide e investiga dichas señales, sin repetir análisis de negocio ni construir visualizaciones finales.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 110)

In [2]:
working_directory = Path.cwd().resolve()
PROJECT_ROOT = working_directory if (working_directory / 'data' / 'raw').exists() else working_directory.parent
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
MOVIES_PATH = DATA_RAW / 'netflix_movies_detailed_up_to_2025.csv'
TV_SHOWS_PATH = DATA_RAW / 'netflix_tv_shows_detailed_up_to_2025.csv'
for dataset_path in (MOVIES_PATH, TV_SHOWS_PATH):
    if not dataset_path.is_file():
        raise FileNotFoundError(f'No se encontró el archivo requerido: {dataset_path}')
movies_df = pd.read_csv(MOVIES_PATH)
tv_shows_df = pd.read_csv(TV_SHOWS_PATH)
datasets = {'Movies': movies_df, 'TV Shows': tv_shows_df}
print(f'Raíz: {PROJECT_ROOT}')
print('Datasets cargados en memoria; las fuentes RAW no se modifican.')

Raíz: C:\Users\cesar\OneDrive\Desktop\DuocUC\3erYear\visualizacionDatos\streamview-analytics-visualizacion-datos
Datasets cargados en memoria; las fuentes RAW no se modifican.


## 2. Perfil de calidad y nulos

Para cada campo se informa tipo, completitud, cardinalidad, unicidad, ejemplos y estadísticos numéricos. La severidad de disponibilidad es: crítica ≥50 %, alta 10–<50 %, media 1–<10 % y baja <1 %; no ordena imputar ni eliminar filas.

In [3]:
def quality_profile(dataframe: pd.DataFrame, source_name: str) -> pd.DataFrame:
    rows = []
    for column in dataframe.columns:
        series = dataframe[column]
        row = {'fuente': source_name, 'variable': column, 'tipo': str(series.dtype), 'registros': len(dataframe), 'no_nulos': int(series.notna().sum()), 'nulos': int(series.isna().sum()), 'porcentaje_nulo': series.isna().mean() * 100, 'valores_unicos': int(series.nunique(dropna=True)), 'porcentaje_unicidad': series.nunique(dropna=True) / len(dataframe) * 100, 'ejemplos': ' | '.join(map(str, series.dropna().drop_duplicates().head(3).tolist()))}
        if pd.api.types.is_numeric_dtype(series):
            row.update({'minimo': series.min(), 'p01': series.quantile(.01), 'p25': series.quantile(.25), 'mediana': series.median(), 'media': series.mean(), 'p75': series.quantile(.75), 'p99': series.quantile(.99), 'maximo': series.max()})
        rows.append(row)
    return pd.DataFrame(rows)

def null_summary(dataframe: pd.DataFrame, source_name: str) -> pd.DataFrame:
    result = pd.DataFrame({'variable': dataframe.columns, 'nulos': dataframe.isna().sum().to_numpy()})
    result['fuente'] = source_name
    result['porcentaje_nulo'] = result['nulos'] / len(dataframe) * 100
    result['severidad_disponibilidad'] = pd.cut(result['porcentaje_nulo'], [-.1, .999999, 9.999999, 49.999999, 100], labels=['baja', 'media', 'alta', 'crítica']).astype(str)
    return result.sort_values(['porcentaje_nulo', 'variable'], ascending=[False, True])

quality_profiles = pd.concat([quality_profile(df, name) for name, df in datasets.items()], ignore_index=True)
nulls = pd.concat([null_summary(df, name) for name, df in datasets.items()], ignore_index=True)
display(quality_profiles)
display(nulls)

,fuente,variable,tipo,registros,no_nulos,nulos,porcentaje_nulo,valores_unicos,porcentaje_unicidad,ejemplos,minimo,p01,p25,mediana,media,p75,p99,maximo
0,Movies,show_id,int64,16000,16000,0,0.00000,16000,100.00000,10192 | 27205 | 12444,189.000,35457.70000,225725.75000,446817.0000,5.266582e+05,7.739808e+05,1.430277e+06,1.440471e+06
1,Movies,type,str,16000,16000,0,0.00000,1,0.00625,Movie,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Movies,title,str,16000,16000,0,0.00000,15485,96.78125,Shrek Forever After | Inception | Harry Potter and the Deathly Hallows: Part 1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Movies,director,str,16000,15868,132,0.82500,9508,59.42500,Mike Mitchell | Christopher Nolan | David Yates,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Movies,cast,str,16000,15796,204,1.27500,15639,97.74375,"Mike Myers, Eddie Murphy, Cameron Diaz, Antonio Banderas, Walt Dohrn | Leonardo DiCaprio, Joseph Gordon-Le...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Movies,country,str,16000,15534,466,2.91250,1463,9.14375,"United States of America | United Kingdom, United States of America | China, Hong Kong, United States of A...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Movies,date_added,str,16000,16000,0,0.00000,4423,27.64375,2010-05-16 | 2010-07-15 | 2010-11-17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Movies,release_year,int64,16000,16000,0,0.00000,16,0.10000,2010 | 2011 | 2012,2010.000,2010.00000,2013.75000,2017.5000,2.017500e+03,2.021250e+03,2.025000e+03,2.025000e+03
8,Movies,rating,float64,16000,16000,0,0.00000,2145,13.40625,6.38 | 8.369 | 7.744,0.000,0.00000,5.60000,6.3000,5.956368e+00,6.923000e+00,8.333000e+00,1.000000e+01
9,Movies,duration,float64,16000,0,16000,100.00000,0,0.00000,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,variable,nulos,fuente,porcentaje_nulo,severidad_disponibilidad
0,duration,16000,Movies,100.00000,crítica
1,country,466,Movies,2.91250,media
2,cast,204,Movies,1.27500,media
3,description,132,Movies,0.82500,baja
4,director,132,Movies,0.82500,baja
5,genres,107,Movies,0.66875,baja
6,budget,0,Movies,0.00000,baja
7,date_added,0,Movies,0.00000,baja
8,language,0,Movies,0.00000,baja
9,popularity,0,Movies,0.00000,baja


**Decisión propuesta:** no aplicar dropna ni imputación global. En análisis futuros se excluyen solo registros sin la variable requerida y se informa cobertura; los nulos permanecen intactos en la copia de trabajo.

## 3. Duplicados exactos y auditoría de show_id

Los duplicados exactos se revisan por separado. Para IDs repetidos se muestran los metadatos completos y se clasifican como filas idénticas, títulos distintos, diferencias menores o otras diferencias. No se elimina ningún registro.

In [4]:
def id_audit(dataframe: pd.DataFrame, source_name: str) -> tuple[dict, pd.DataFrame, pd.DataFrame]:
    exact_mask = dataframe.duplicated(keep=False)
    repeated = dataframe[dataframe['show_id'].duplicated(keep=False)].copy()
    groups = []
    for show_id, group in repeated.groupby('show_id', dropna=False):
        comparable = group.astype(object).where(group.notna(), '<NA>')
        varying = [column for column in group.columns if comparable[column].nunique(dropna=False) > 1]
        classification = 'filas idénticas' if not varying else ('títulos distintos' if group['title'].nunique(dropna=False) > 1 else ('diferencias menores de metadatos' if set(varying).issubset({'description', 'popularity', 'vote_count', 'vote_average', 'rating'}) else 'otras diferencias de metadatos'))
        groups.append({'fuente': source_name, 'show_id': show_id, 'filas': len(group), 'clasificacion': classification, 'campos_con_diferencias': ', '.join(varying) if varying else 'ninguno'})
    summary = {'fuente': source_name, 'duplicados_exactos_incluyendo_repeticiones': int(exact_mask.sum()), 'porcentaje_duplicados_exactos': exact_mask.mean() * 100, 'show_id_unicos': int(dataframe['show_id'].nunique(dropna=True)), 'ids_repetidos': int(repeated['show_id'].nunique(dropna=True)), 'filas_con_id_repetido': len(repeated), 'porcentaje_filas_con_id_repetido': len(repeated) / len(dataframe) * 100}
    fields = ['show_id', 'title', 'type', 'release_year', 'director', 'cast', 'country', 'genres', 'language', 'popularity', 'vote_average', 'vote_count', 'rating', 'duration', 'date_added', 'description']
    return summary, pd.DataFrame(groups), repeated[fields].sort_values(['show_id', 'title'])

id_results = [id_audit(df, name) for name, df in datasets.items()]
id_summary = pd.DataFrame([result[0] for result in id_results])
id_classes = pd.concat([result[1] for result in id_results], ignore_index=True)
id_details = pd.concat([result[2] for result in id_results], ignore_index=True)
display(id_summary)
display(id_classes)
display(id_details)

,fuente,duplicados_exactos_incluyendo_repeticiones,porcentaje_duplicados_exactos,show_id_unicos,ids_repetidos,filas_con_id_repetido,porcentaje_filas_con_id_repetido
0,Movies,0,0.0,16000,0,0,0.0000
1,TV Shows,0,0.0,15991,9,18,0.1125


,fuente,show_id,filas,clasificacion,campos_con_diferencias
0,TV Shows,252630,2,diferencias menores de metadatos,popularity
1,TV Shows,278867,2,diferencias menores de metadatos,popularity
2,TV Shows,279457,2,diferencias menores de metadatos,popularity
3,TV Shows,279739,2,diferencias menores de metadatos,popularity
4,TV Shows,280651,2,diferencias menores de metadatos,popularity
5,TV Shows,281750,2,diferencias menores de metadatos,popularity
6,TV Shows,283602,2,diferencias menores de metadatos,popularity
7,TV Shows,283929,2,diferencias menores de metadatos,popularity
8,TV Shows,284416,2,diferencias menores de metadatos,popularity


,show_id,title,type,release_year,director,cast,country,genres,language,popularity,vote_average,vote_count,rating,duration,date_added,description
0,252630,Joe Lycett’s United States of Birmingham,TV Show,2025,Nicola Silk,Joe Lycett,United Kingdom,"Documentary, Comedy",en,11.550,0.0,0,0.0,1 Seasons,2025-03-01,"It’s no secret that Joe Lycett adores Brum, but his hometown is having a tough time of it. In his self-app..."
1,252630,Joe Lycett’s United States of Birmingham,TV Show,2025,Nicola Silk,Joe Lycett,United Kingdom,"Documentary, Comedy",en,11.531,0.0,0,0.0,1 Seasons,2025-03-01,"It’s no secret that Joe Lycett adores Brum, but his hometown is having a tough time of it. In his self-app..."
2,278867,Ensemble,TV Show,2025,Hayato Kawai,"Haruna Kawaguchi, Hokuto Matsumura, Neru Nagahama, Jiro, Yuka Itaya",Japan,"Drama, Mystery",ja,11.044,7.0,1,7.0,1 Seasons,2025-01-18,"Lawyer Sena Koyama, a realist disillusioned by love, teams up with idealistic newcomer Yu Matohara, who be..."
3,278867,Ensemble,TV Show,2025,Hayato Kawai,"Haruna Kawaguchi, Hokuto Matsumura, Neru Nagahama, Jiro, Yuka Itaya",Japan,"Drama, Mystery",ja,18.407,7.0,1,7.0,1 Seasons,2025-01-18,"Lawyer Sena Koyama, a realist disillusioned by love, teams up with idealistic newcomer Yu Matohara, who be..."
4,279457,Ghassan & Mina,TV Show,2025,NaN,"Razzaq Ahmed, Sulaf Jalil",NaN,Comedy,ar,9.631,0.0,0,0.0,1 Seasons,2025-03-01,NaN
5,279457,Ghassan & Mina,TV Show,2025,NaN,"Razzaq Ahmed, Sulaf Jalil",NaN,Comedy,ar,16.051,0.0,0,0.0,1 Seasons,2025-03-01,NaN
6,279739,Settle Down,TV Show,2025,NaN,"Alexander Nunez, Tymika Tafari, Nadine Bhabha, Izad Etemadi, Leighton Alexander Williams",Canada,NaN,en,15.215,0.0,0,0.0,1 Seasons,2025-02-14,Mason is an expert on queer relationships and a matchmaker with a booming success rate. When it comes to l...
7,279739,Settle Down,TV Show,2025,NaN,"Alexander Nunez, Tymika Tafari, Nadine Bhabha, Izad Etemadi, Leighton Alexander Williams",Canada,NaN,en,13.135,0.0,0,0.0,1 Seasons,2025-02-14,Mason is an expert on queer relationships and a matchmaker with a booming success rate. When it comes to l...
8,280651,Master in 3 Months: Edo Period,TV Show,2025,NaN,"Hiroyuki Nojima, Noriko Hidaka",Japan,Talk,ja,16.794,0.0,0,0.0,1 Seasons,2025-01-08,An easy-to-understand explanation of the Edo Period.
9,280651,Master in 3 Months: Edo Period,TV Show,2025,NaN,"Hiroyuki Nojima, Noriko Hidaka",Japan,Talk,ja,11.996,0.0,0,0.0,1 Seasons,2025-01-08,An easy-to-understand explanation of the Edo Period.


### 3.1 Evidencia compacta de los pares repetidos de TV Shows

Cada par se compara sin modificarlo. La diferencia relativa se calcula respecto de popularity del registro A solo cuando ese valor es distinto de cero. La tabla permite comprobar formalmente si todas las columnas, salvo popularity, coinciden.


In [5]:
def repeated_id_pair_evidence(dataframe: pd.DataFrame) -> pd.DataFrame:
    repeated = dataframe.loc[dataframe['show_id'].duplicated(keep=False)].copy()
    rows = []
    for show_id, group in repeated.groupby('show_id', sort=True):
        first, second = group.iloc[0], group.iloc[1]
        first_values = first.astype(object).where(first.notna(), '<NA>')
        second_values = second.astype(object).where(second.notna(), '<NA>')
        different_columns = first.index[first_values.ne(second_values)].tolist()
        popularity_a, popularity_b = first['popularity'], second['popularity']
        difference_absolute = abs(popularity_b - popularity_a)
        difference_relative = (difference_absolute / abs(popularity_a) * 100) if popularity_a != 0 else np.nan
        rows.append({
            'show_id': show_id,
            'title': first['title'],
            'popularity_registro_a': popularity_a,
            'popularity_registro_b': popularity_b,
            'diferencia_absoluta_popularity': difference_absolute,
            'diferencia_relativa_porcentaje': difference_relative,
            'cantidad_columnas_diferentes': len(different_columns),
            'columnas_diferentes': ', '.join(different_columns),
            'mismo_titulo': first['title'] == second['title'],
            'solo_popularity_cambia': different_columns == ['popularity'],
        })
    return pd.DataFrame(rows)

repeated_id_pairs = repeated_id_pair_evidence(tv_shows_df)
display(repeated_id_pairs)
print(f'Pares revisados: {len(repeated_id_pairs)}')
print(f'Pares con mismo título: {int(repeated_id_pairs["mismo_titulo"].sum())} de {len(repeated_id_pairs)}')
print(f'Pares donde solo cambia popularity: {int(repeated_id_pairs["solo_popularity_cambia"].sum())} de {len(repeated_id_pairs)}')

,show_id,title,popularity_registro_a,popularity_registro_b,diferencia_absoluta_popularity,diferencia_relativa_porcentaje,cantidad_columnas_diferentes,columnas_diferentes,mismo_titulo,solo_popularity_cambia
0,252630,Joe Lycett’s United States of Birmingham,11.550,11.531,0.019,0.164502,1,popularity,True,True
1,278867,Ensemble,11.044,18.407,7.363,66.669685,1,popularity,True,True
2,279457,Ghassan & Mina,9.631,16.051,6.420,66.659745,1,popularity,True,True
3,279739,Settle Down,15.215,13.135,2.080,13.670720,1,popularity,True,True
4,280651,Master in 3 Months: Edo Period,16.794,11.996,4.798,28.569727,1,popularity,True,True
5,281750,الباء تحته نقطة,13.448,18.393,4.945,36.771267,1,popularity,True,True
6,283602,رحمة,22.466,16.047,6.419,28.572064,1,popularity,True,True
7,283929,Kimi wa Mendou na Konyakusha,24.577,21.172,3.405,13.854417,1,popularity,True,True
8,284416,Par de ideotas,14.680,14.553,0.127,0.865123,1,popularity,True,True


Pares revisados: 9
Pares con mismo título: 9 de 9
Pares donde solo cambia popularity: 9 de 9


**Interpretación:** si la última columna es verdadera en los nueve pares, el problema no es un duplicado exacto: es un **conflicto de atributo para una misma entidad identificada por show_id**. Para conteo de catálogo, ambas filas no deben representar dos contenidos. Para análisis de popularity, la fuente no permite decidir cuál de los dos valores es correcto; no se conserva primero/último, máximo/mínimo ni se calcula un promedio en esta rama.


### 3.2 Impacto diagnostico de la politica de popularity

La decision metodologica es excluir los show_id conflictivos unicamente de calculos, KPIs, rankings y graficos que dependan directamente de popularity. No se excluyen de analisis que no usan popularity. La deteccion se deriva programaticamente del conflicto observado; la futura columna booleana popularity_conflict pertenece a cesar/feature/preparacion-catalogo y no se materializa en esta rama.


In [6]:
conflicting_show_ids = repeated_id_pairs.loc[repeated_id_pairs['solo_popularity_cambia'], 'show_id']
conflicting_rows = tv_shows_df['show_id'].isin(conflicting_show_ids).sum()
unique_entities_tv = tv_shows_df['show_id'].nunique(dropna=True)
top_10_threshold = tv_shows_df['popularity'].nlargest(10).iloc[-1]
maximum_conflicting_popularity = tv_shows_df.loc[tv_shows_df['show_id'].isin(conflicting_show_ids), 'popularity'].max()
popularity_policy_impact = pd.DataFrame([{
    'ids_conflictivos': int(conflicting_show_ids.nunique()),
    'filas_afectadas': int(conflicting_rows),
    'entidades_unicas_tv_shows': int(unique_entities_tv),
    'porcentaje_entidades_afectadas': conflicting_show_ids.nunique() / unique_entities_tv * 100,
    'umbral_aproximado_top_10_popularity': top_10_threshold,
    'maxima_popularity_en_conflicto': maximum_conflicting_popularity,
    'conflictos_alcanzan_umbral_top_10': maximum_conflicting_popularity >= top_10_threshold,
}])
display(popularity_policy_impact)
print('Diagnóstico: la política no crea ni persiste popularity_conflict en este notebook.')

,ids_conflictivos,filas_afectadas,entidades_unicas_tv_shows,porcentaje_entidades_afectadas,umbral_aproximado_top_10_popularity,maxima_popularity_en_conflicto,conflictos_alcanzan_umbral_top_10
0,9,18,15991,0.056282,2379.884,24.577,False


Diagnóstico: la política no crea ni persiste popularity_conflict en este notebook.


**Interpretacion:** el porcentaje se calcula sobre entidades unicas, no sobre las 18 filas. La comparacion con el umbral Top 10 solo estima si la exclusion podria afectar rankings altos; no es una conclusion de negocio. La preparacion debera volver a detectar estos IDs, crear popularity_conflict, evitar doble conteo de entidad y excluirlos solamente de analisis dependientes de popularity, preservando ambas filas RAW para trazabilidad.


In [7]:
def raw_id_semantics(path: Path, source_name: str) -> dict:
    raw_ids = pd.read_csv(path, usecols=['show_id'], dtype={'show_id': 'string'})['show_id']
    return {'fuente': source_name, 'tipo_cargado': str(datasets[source_name]['show_id'].dtype), 'nulos_raw': int(raw_ids.isna().sum()), 'con_ceros_iniciales': int(raw_ids.str.match(r'^0+\d+$', na=False).sum()), 'no_numericos': int((~raw_ids.str.fullmatch(r'\d+', na=False) & raw_ids.notna()).sum()), 'longitud_min': raw_ids.str.len().min(), 'longitud_max': raw_ids.str.len().max()}
id_semantics = pd.DataFrame([raw_id_semantics(MOVIES_PATH, 'Movies'), raw_id_semantics(TV_SHOWS_PATH, 'TV Shows')])
id_semantics

,fuente,tipo_cargado,nulos_raw,con_ceros_iniciales,no_numericos,longitud_min,longitud_max
0,Movies,int64,0,0,0,3,7
1,TV Shows,int64,0,0,0,3,6


**Decisión propuesta:** no deduplicar por show_id en esta rama. Conservar el identificador RAW y convertir solo la copia analítica a texto en preparación, después de documentar una política para cada patrón repetido.

## 4. Tipos, formatos y categorías

Se valida la inconsistencia de rating, la falta de otro campo explícitamente etario, duration y la higiene de campos categóricos. Los resultados describen formatos; no normalizan mayúsculas, espacios ni separadores.

In [8]:
rating_rows = []
for name, dataframe in datasets.items():
    comparable = dataframe['rating'].notna() & dataframe['vote_average'].notna()
    matches = dataframe['rating'].eq(dataframe['vote_average']) & comparable
    rating_rows.append({'fuente': name, 'tipo_rating': str(dataframe['rating'].dtype), 'comparables': int(comparable.sum()), 'coincidencias_con_vote_average': int(matches.sum()), 'porcentaje_coincidencia': matches.sum() / comparable.sum() * 100, 'rating_fuera_0_10': int((dataframe['rating'].lt(0) | dataframe['rating'].gt(10)).sum()), 'duration_nulos': int(dataframe['duration'].isna().sum()), 'duration_valores_unicos_no_nulos': int(dataframe['duration'].nunique(dropna=True)), 'ejemplos_duration': ' | '.join(map(str, dataframe['duration'].dropna().drop_duplicates().head(3)))})
rating_duration_audit = pd.DataFrame(rating_rows)
age_named_columns = [column for column in sorted(set(movies_df.columns) | set(tv_shows_df.columns)) if any(token in column.lower() for token in ('age', 'certificate', 'classification', 'maturity'))]
display(rating_duration_audit)
print('Columnas adicionales cuyo nombre sugiere clasificación etaria:', age_named_columns)

,fuente,tipo_rating,comparables,coincidencias_con_vote_average,porcentaje_coincidencia,rating_fuera_0_10,duration_nulos,duration_valores_unicos_no_nulos,ejemplos_duration
0,Movies,float64,16000,16000,100.0,0,16000,0,
1,TV Shows,float64,16000,16000,100.0,0,0,1,1 Seasons


Columnas adicionales cuyo nombre sugiere clasificación etaria: ['language', 'vote_average']


In [9]:
CATEGORICAL_COLUMNS = ['type', 'country', 'genres', 'language', 'director', 'cast']
PLACEHOLDERS = {'', 'na', 'n/a', 'none', 'null', 'unknown', 'sin información', 'sin informacion'}
def categorical_quality(dataframe: pd.DataFrame, source_name: str, column: str) -> dict:
    series = dataframe[column].dropna().astype(str)
    stripped = series.str.strip()
    variants = stripped.groupby(stripped.str.casefold()).nunique()
    return {'fuente': source_name, 'variable': column, 'no_nulos': len(series), 'vacios_o_espacios': int(stripped.eq('').sum()), 'espacios_inicio_fin': int(series.ne(stripped).sum()), 'placeholders': int(stripped.str.casefold().isin(PLACEHOLDERS).sum()), 'variantes_solo_por_mayusculas': int((variants > 1).sum()), 'con_coma': int(series.str.contains(',', regex=False).sum()), 'con_punto_y_coma': int(series.str.contains(';', regex=False).sum()), 'con_pipe': int(series.str.contains('|', regex=False).sum()), 'ejemplos': ' | '.join(stripped.drop_duplicates().head(5).tolist())}
categorical_audit = pd.DataFrame([categorical_quality(df, name, column) for name, df in datasets.items() for column in CATEGORICAL_COLUMNS])
categorical_audit

,fuente,variable,no_nulos,vacios_o_espacios,espacios_inicio_fin,placeholders,variantes_solo_por_mayusculas,con_coma,con_punto_y_coma,con_pipe,ejemplos
0,Movies,type,16000,0,0,0,0,0,0,0,Movie
1,Movies,country,15534,0,0,0,0,4058,0,0,"United States of America | United Kingdom, United States of America | China, Hong Kong, United States of A..."
2,Movies,genres,15893,0,0,0,0,12116,0,0,"Comedy, Adventure, Fantasy, Animation, Family | Action, Science Fiction, Adventure | Adventure, Fantasy | ..."
3,Movies,language,16000,0,0,0,0,0,0,0,en | es | sr | pt | ko
4,Movies,director,15868,0,1,0,1,1286,0,0,"Mike Mitchell | Christopher Nolan | David Yates | Byron Howard, Nathan Greno | Chris Sanders, Dean DeBlois"
5,Movies,cast,15796,0,2,0,0,15632,0,0,"Mike Myers, Eddie Murphy, Cameron Diaz, Antonio Banderas, Walt Dohrn | Leonardo DiCaprio, Joseph Gordon-Le..."
6,TV Shows,type,16000,0,0,0,0,0,0,0,TV Show
7,TV Shows,country,14203,0,0,0,0,1307,0,0,"South Korea | United States of America | Greece | Czech Republic | United States of America, Ireland"
8,TV Shows,genres,15026,0,0,1,0,8696,0,0,"Comedy, Reality | Talk, Comedy, News | Reality | Talk | Drama, Soap"
9,TV Shows,language,16000,0,0,0,0,0,0,0,ko | en | el | cs | ru


## 5. Campos multivalor

La auditoría comprueba si la coma funciona como separador, si hay espacios inconsistentes después de ella, elementos vacíos, elementos repetidos dentro de una celda y tamaños de lista. No se explotan tablas en esta etapa.

In [10]:
MULTIVALUE_COLUMNS = ['country', 'genres', 'director', 'cast']
def multivalue_quality(dataframe: pd.DataFrame, source_name: str, column: str) -> dict:
    series = dataframe[column].dropna().astype(str)
    comma_mask = series.str.contains(',', regex=False)
    comma_values = series[comma_mask]
    lists = comma_values.map(lambda value: [piece.strip() for piece in value.split(',')])
    item_counts = lists.map(len)
    duplicate_elements = lists.map(lambda values: len([value for value in values if value]) != len(set(value.casefold() for value in values if value)))
    return {'fuente': source_name, 'variable': column, 'celdas_con_coma': int(comma_mask.sum()), 'porcentaje_con_coma': comma_mask.mean() * 100, 'coma_sin_espacio_posterior': int(comma_values.str.contains(r',\S', regex=True).sum()), 'elementos_vacios': int(lists.map(lambda values: any(not value for value in values)).sum()), 'elementos_repetidos_en_celda': int(duplicate_elements.sum()), 'items_mediana': item_counts.median() if len(item_counts) else np.nan, 'items_p90': item_counts.quantile(.9) if len(item_counts) else np.nan, 'items_maximo': item_counts.max() if len(item_counts) else np.nan}
multivalue_audit = pd.DataFrame([multivalue_quality(df, name, column) for name, df in datasets.items() for column in MULTIVALUE_COLUMNS])
display(multivalue_audit)
print('language se mantiene como categórica simple en esta auditoría: no se propone separarla sin evidencia adicional.')

,fuente,variable,celdas_con_coma,porcentaje_con_coma,coma_sin_espacio_posterior,elementos_vacios,elementos_repetidos_en_celda,items_mediana,items_p90,items_maximo
0,Movies,country,4058,26.123342,0,0,0,2.0,4.0,16
1,Movies,genres,12116,76.234820,0,0,0,3.0,4.0,8
2,Movies,director,1286,8.104361,0,0,0,2.0,3.0,31
3,Movies,cast,15632,98.961762,0,0,8,5.0,5.0,6
4,TV Shows,country,1307,9.202281,0,0,0,2.0,3.0,12
5,TV Shows,genres,8696,57.873020,0,0,13,2.0,4.0,7
6,TV Shows,director,1321,26.236346,0,0,4,2.0,4.0,11
7,TV Shows,cast,13353,89.961598,0,0,26,5.0,5.0,6


language se mantiene como categórica simple en esta auditoría: no se propone separarla sin evidencia adicional.


## 6. Fechas, dominios y atípicos

date_added se convierte solo en una serie temporal diagnóstica. La columna original permanece como texto. Los atípicos se cuantifican mediante percentiles e IQR; un atípico no equivale a error ni se elimina automáticamente.

In [11]:
date_rows = []
for name, dataframe in datasets.items():
    parsed = pd.to_datetime(dataframe['date_added'], errors='coerce')
    invalid = dataframe['date_added'].notna() & parsed.isna()
    prior_release = parsed.dt.year < dataframe['release_year']
    date_rows.append({'fuente': name, 'tipo_original': str(dataframe['date_added'].dtype), 'nulos': int(dataframe['date_added'].isna().sum()), 'parseables': int(parsed.notna().sum()), 'no_parseables': int(invalid.sum()), 'fecha_minima_diagnostica': parsed.min(), 'fecha_maxima_diagnostica': parsed.max(), 'date_added_anterior_a_release_year': int(prior_release.sum())})
date_audit = pd.DataFrame(date_rows)

def domain_summary(dataframe: pd.DataFrame, source_name: str, column: str, lower: float | None = None, upper: float | None = None) -> dict:
    series = dataframe[column]
    return {'fuente': source_name, 'variable': column, 'nulos': int(series.isna().sum()), 'negativos': int(series.lt(0).sum()), 'ceros': int(series.eq(0).sum()), 'fuera_limite_inferior': int(series.lt(lower).sum()) if lower is not None else np.nan, 'fuera_limite_superior': int(series.gt(upper).sum()) if upper is not None else np.nan, 'minimo': series.min(), 'p99': series.quantile(.99), 'maximo': series.max()}
def iqr_summary(dataframe: pd.DataFrame, source_name: str, column: str) -> dict:
    series = dataframe[column].dropna()
    q1, q3 = series.quantile(.25), series.quantile(.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return {'fuente': source_name, 'variable': column, 'p01': series.quantile(.01), 'p25': q1, 'mediana': series.median(), 'p75': q3, 'p99': series.quantile(.99), 'iqr': iqr, 'limite_inferior_iqr': lower, 'limite_superior_iqr': upper, 'fuera_iqr': int(((series < lower) | (series > upper)).sum())}
domain_audit = pd.DataFrame([domain_summary(df, name, 'vote_average', 0, 10) for name, df in datasets.items()] + [domain_summary(df, name, column) for name, df in datasets.items() for column in ['vote_count', 'popularity']])
outlier_audit = pd.DataFrame([iqr_summary(df, name, column) for name, df in datasets.items() for column in ['popularity', 'vote_average', 'vote_count']] + [iqr_summary(movies_df, 'Movies', column) for column in ['budget', 'revenue']])
display(date_audit)
display(domain_audit)
display(outlier_audit)

,fuente,tipo_original,nulos,parseables,no_parseables,fecha_minima_diagnostica,fecha_maxima_diagnostica,date_added_anterior_a_release_year
0,Movies,str,0,16000,0,2010-01-01,2025-12-25,0
1,TV Shows,str,0,16000,0,2010-01-01,2025-12-31,0


,fuente,variable,nulos,negativos,ceros,fuera_limite_inferior,fuera_limite_superior,minimo,p99,maximo
0,Movies,vote_average,0,0,899,0.0,0.0,0.000,8.33300,10.000
1,TV Shows,vote_average,0,0,3673,0.0,0.0,0.000,10.00000,10.000
2,Movies,vote_count,0,0,894,NaN,NaN,0.000,10616.24000,37119.000
3,Movies,popularity,0,0,0,NaN,NaN,3.860,144.48478,3876.006
4,TV Shows,vote_count,0,0,3674,NaN,NaN,0.000,1817.05000,24664.000
5,TV Shows,popularity,0,0,0,NaN,NaN,2.323,467.67606,6421.923


,fuente,variable,p01,p25,mediana,p75,p99,iqr,limite_inferior_iqr,limite_superior_iqr,fuera_iqr
0,Movies,popularity,4.53199,7.84075,10.9135,1.733650e+01,1.444848e+02,9.495750e+00,-6.402875e+00,3.158013e+01,1705
1,Movies,vote_average,0.00000,5.60000,6.3000,6.923000e+00,8.333000e+00,1.323000e+00,3.615500e+00,8.907500e+00,1172
2,Movies,vote_count,0.00000,53.00000,138.0000,4.220000e+02,1.061624e+04,3.690000e+02,-5.005000e+02,9.755000e+02,2287
3,TV Shows,popularity,5.40596,24.87475,36.1875,6.218750e+01,4.676761e+02,3.731275e+01,-3.109438e+01,1.181566e+02,1609
4,TV Shows,vote_average,0.00000,2.70000,6.8000,7.800000e+00,1.000000e+01,5.100000e+00,-4.950000e+00,1.545000e+01,0
5,TV Shows,vote_count,0.00000,1.00000,4.0000,3.000000e+01,1.817050e+03,2.900000e+01,-4.250000e+01,7.350000e+01,2661
6,Movies,budget,0.00000,0.00000,0.0000,2.200000e+06,1.650500e+08,2.200000e+06,-3.300000e+06,5.500000e+06,3195
7,Movies,revenue,0.00000,0.00000,0.0000,1.654473e+06,5.432866e+08,1.654473e+06,-2.481710e+06,4.136182e+06,3372


### Clasificación de valores extremos

1. **Extremo estadístico:** queda fuera de 1,5 × IQR; solo describe su posición relativa en la distribución.
2. **Valor inválido por dominio:** viola una regla conocida, por ejemplo un negativo donde no corresponde, vote_average fuera de 0–10 o una fecha no interpretable.
3. **Valor sospechoso:** requiere investigación adicional, pero aún no existe evidencia para declararlo erróneo.

**Política:** un extremo estadístico no equivale a un dato erróneo. Se conservan valores plausibles de popularity, vote_count, budget y revenue; se investigan violaciones de dominio. La preparación o visualización futura podrá usar escala logarítmica, límites visuales u otros filtros documentados, pero nunca borrado automático por IQR.

## 7. Auditoría financiera de Movies

budget y revenue existen solo en Movies. Se mide disponibilidad, ceros, negativos, magnitud y combinaciones de disponibilidad. No se calcula ROI ni se interpreta un cero como falta de información sin evidencia externa.

In [12]:
finance_rows = []
for column in ['budget', 'revenue']:
    series = movies_df[column]
    finance_rows.append({'variable': column, 'tipo': str(series.dtype), 'nulos': int(series.isna().sum()), 'ceros': int(series.eq(0).sum()), 'negativos': int(series.lt(0).sum()), 'minimo': series.min(), 'p01': series.quantile(.01), 'mediana': series.median(), 'p99': series.quantile(.99), 'maximo': series.max(), 'fuera_iqr': int(outlier_audit.loc[outlier_audit['variable'].eq(column), 'fuera_iqr'].iloc[0])})
financial_categories = pd.Series(np.select([movies_df['budget'].gt(0) & movies_df['revenue'].gt(0), movies_df['budget'].eq(0) & movies_df['revenue'].gt(0), movies_df['budget'].gt(0) & movies_df['revenue'].eq(0), movies_df['budget'].eq(0) & movies_df['revenue'].eq(0), movies_df['budget'].lt(0) | movies_df['revenue'].lt(0)], ['budget > 0 y revenue > 0', 'budget cero y revenue > 0', 'budget > 0 y revenue cero', 'ambos cero', 'al menos un negativo'], default='otros / nulos'), name='categoria_financiera').value_counts().rename_axis('categoria_financiera').reset_index(name='registros')
display(pd.DataFrame(finance_rows))
display(financial_categories)

,variable,tipo,nulos,ceros,negativos,minimo,p01,mediana,p99,maximo,fuera_iqr
0,budget,int64,0,11153,0,0,0.0,0.0,1.650500e+08,460000000,3195
1,revenue,int64,0,10355,0,0,0.0,0.0,5.432866e+08,2799439100,3372


,categoria_financiera,registros
0,ambos cero,9048
1,budget > 0 y revenue > 0,3540
2,budget cero y revenue > 0,2105
3,budget > 0 y revenue cero,1307


## 8. Conclusión y handoff

### Problemas corregibles mediante preparación

- date_added puede convertirse de forma reproducible a datetime; show_id puede representarse semánticamente como texto.
- Los campos multivalor y los pocos espacios de texto pueden prepararse mediante reglas documentadas y copias trazables.

### Limitaciones permanentes de la fuente

- rating no aporta clasificación etaria confiable; duration no aporta duración útil; la ausencia de director en TV Shows no puede repararse sin inventar información.

### Decisiones metodologicas ya tomadas

- Los nueve show_id repetidos son conflictos de popularity: para conteo representan una entidad unica; para cualquier analisis dependiente de popularity se excluyen ambas filas del ID conflictivo. No se elige primero, ultimo, maximo, minimo ni promedio.

### Decisiones todavia pendientes

- Los ceros financieros, el tratamiento analitico de nulos y el uso visual de extremos requieren una politica explicita.
- Los outliers estadísticos no se trataron como errores automáticos; solo las violaciones de dominio justifican investigación de invalidez.

### Handoff a cesar/feature/preparacion-catalogo

Aplicar, solo despues de aprobar estas decisiones: representacion semantica de show_id como texto, conversion definitiva de date_added, reglas documentadas de espacios/categorias, separacion por coma y tablas auxiliares para country, genres, director y cast; ademas de filtros explicitos y reproducibles para analisis financiero. Detectar programaticamente los nueve IDs conflictivos, crear popularity_conflict, evitar doble conteo de la entidad y construir la version apta para analisis de popularity excluyendo ambos registros de esos IDs; documentar la cantidad de entidades excluidas y preservar las filas RAW. Los RAW permanecen intactos.

In [13]:
print('Auditoría finalizada sin modificaciones persistentes sobre los CSV RAW.')
print('Consulte docs/calidad_datos_decisiones.md para la matriz de decisiones propuesta.')

Auditoría finalizada sin modificaciones persistentes sobre los CSV RAW.
Consulte docs/calidad_datos_decisiones.md para la matriz de decisiones propuesta.
